# Extraction de donnees structurees depuis des CV

Ce notebook presente une demonstration pas a pas de l'extraction
d'informations structurees a partir de CV au format texte brut.

## Approche

Nous utilisons les **expressions regulieres** (module `re` de Python) pour :
1. Identifier les differentes sections du CV
2. Extraire les informations de contact (nom, email, telephone)
3. Parser les competences, experiences et formations

> **Note** : Pour un usage en production, on pourrait utiliser spaCy avec
> un modele NER (Named Entity Recognition) pour une extraction plus robuste.

## 1. Importation des modules

In [ ]:
import re
import json
import sys
from pathlib import Path

# Ajouter le dossier parent au path pour importer notre module
sys.path.insert(0, str(Path('.').resolve().parent))

from src.extractor import (
    extraire_nom,
    extraire_email,
    extraire_telephone,
    extraire_competences,
    extraire_experience,
    extraire_formation,
    extraire_langues,
    extraire_donnees_cv
)

## 2. Chargement d'un CV exemple

In [ ]:
# Charger le premier CV (data scientist francais)
chemin_cv = Path('..') / 'data' / 'cv1.txt'
with open(chemin_cv, 'r', encoding='utf-8') as f:
    texte_cv = f.read()

# Afficher les premieres lignes
print("=" * 50)
print("CONTENU DU CV (premieres 30 lignes)")
print("=" * 50)
lignes = texte_cv.split('\n')
for ligne in lignes[:30]:
    print(ligne)

## 3. Extraction pas a pas

### 3.1 Extraction du nom

In [ ]:
nom = extraire_nom(texte_cv)
print(f"Nom extrait : {nom}")

# Explication du pattern utilise
print("\n--- Pattern utilise ---")
print("Regex : r'(?:Nom|NOM)\\s*:\\s*(.+)'")
print("Recherche 'Nom :' ou 'NOM :' suivi du contenu")

### 3.2 Extraction de l'email

In [ ]:
email = extraire_email(texte_cv)
print(f"Email extrait : {email}")

# Explication du pattern
print("\n--- Pattern utilise ---")
print("Regex : r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}'")
print("Pattern standard pour detecter une adresse email")

### 3.3 Extraction du telephone

In [ ]:
telephone = extraire_telephone(texte_cv)
print(f"Telephone extrait : {telephone}")

print("\n--- Patterns utilises ---")
print("1. Format 'Telephone : NUMERO'")
print("2. Format international +33 ...")
print("3. Format francais 06 12 34 56 78")

### 3.4 Extraction des competences

In [ ]:
competences = extraire_competences(texte_cv)
print(f"Nombre de competences extraites : {len(competences)}")
print("\nListe des competences :")
for i, comp in enumerate(competences, 1):
    print(f"  {i:2d}. {comp}")

### 3.5 Extraction des experiences professionnelles

In [ ]:
experiences = extraire_experience(texte_cv)
print(f"Nombre d'experiences extraites : {len(experiences)}")
print()
for i, exp in enumerate(experiences, 1):
    print(f"--- Experience {i} ---")
    print(f"  Periode    : {exp['periode']}")
    print(f"  Poste      : {exp['poste']}")
    print(f"  Entreprise : {exp['entreprise']}")
    if exp['description']:
        print(f"  Description :")
        for desc in exp['description']:
            print(f"    - {desc}")
    print()

### 3.6 Extraction des formations

In [ ]:
formations = extraire_formation(texte_cv)
print(f"Nombre de formations extraites : {len(formations)}")
print()
for i, form in enumerate(formations, 1):
    print(f"--- Formation {i} ---")
    print(f"  Periode       : {form['periode']}")
    print(f"  Diplome       : {form['diplome']}")
    print(f"  Etablissement : {form['etablissement']}")
    print()

### 3.7 Extraction des langues

In [ ]:
langues = extraire_langues(texte_cv)
print(f"Nombre de langues extraites : {len(langues)}")
print()
for langue in langues:
    print(f"  {langue['langue']:12s} : {langue['niveau']}")

## 4. Extraction complete

Utilisons la fonction principale pour tout extraire d'un coup :

In [ ]:
# Extraction complete
resultat = extraire_donnees_cv(texte_cv)

# Affichage en JSON formate
print(json.dumps(resultat, ensure_ascii=False, indent=2))

## 5. Test sur un CV en anglais

Verifions que notre extracteur fonctionne aussi sur un CV en anglais :

In [ ]:
# Charger le CV en anglais
chemin_cv3 = Path('..') / 'data' / 'cv3.txt'
with open(chemin_cv3, 'r', encoding='utf-8') as f:
    texte_cv3 = f.read()

# Extraction
resultat_en = extraire_donnees_cv(texte_cv3)

print("Resultat pour le CV en anglais :")
print(f"  Nom   : {resultat_en['nom']}")
print(f"  Email : {resultat_en['email']}")
print(f"  Tel   : {resultat_en['telephone']}")
print(f"  Competences : {len(resultat_en['competences'])} extraites")
print(f"  Experiences : {len(resultat_en['experience_professionnelle'])} postes")
print(f"  Formations  : {len(resultat_en['formation'])} diplomes")
print(f"  Langues     : {len(resultat_en['langues'])} langues")

## 6. Ameliorations possibles

### Avec spaCy (NER)

```python
import spacy

# Charger un modele francais
nlp = spacy.load('fr_core_news_md')

doc = nlp(texte_cv)
for ent in doc.ents:
    print(f"{ent.text:30s} -> {ent.label_}")
```

### Avantages de spaCy :
- Detection automatique des entites nommees (personnes, organisations, lieux)
- Meilleure gestion des variations dans le format
- Modeles pre-entraines pour le francais (fr_core_news_md)

### Avec des Transformers (Hugging Face) :
- CamemBERT pour le francais
- Fine-tuning sur des donnees de CV annotees
- Meilleure comprehension contextuelle

## 7. Evaluation de la qualite de l'extraction

### Methodologie

Pour evaluer la qualite de notre extracteur, nous comparons les resultats obtenus
avec une **verite terrain** (ground truth) construite manuellement a partir des CV.

Les metriques utilisees sont :
- **Precision** : proportion d'elements extraits qui sont corrects
- **Rappel** : proportion d'elements attendus qui ont ete trouves
- **F1-Score** : moyenne harmonique de la precision et du rappel

Pour les **champs simples** (nom, email, telephone) : correspondance exacte (1 ou 0).

Pour les **listes** (competences) : calcul base sur les elements trouves vs attendus.

Pour les **listes de dictionnaires** (experience, formation, langues) : comparaison
sur les champs cles (periode, poste, entreprise, etc.).

In [ ]:
# Charger les resultats extraits et la verite terrain
chemin_resultats = Path('..') / 'results' / 'output.json'
chemin_verite = Path('..') / 'evaluation' / 'ground_truth.json'

with open(chemin_resultats, 'r', encoding='utf-8') as f:
    resultats = json.load(f)

with open(chemin_verite, 'r', encoding='utf-8') as f:
    verite_terrain = json.load(f)

print(f"Nombre de CV dans les resultats : {len(resultats)}")
print(f"Nombre de CV dans la verite terrain : {len(verite_terrain)}")

In [ ]:
# Importer le module d'evaluation
sys.path.insert(0, str(Path('..').resolve() / 'evaluation'))
from evaluate import evaluer_cv, calculer_score_global

# Evaluer chaque CV
resultats_evaluation = {}
for cv_name in sorted(verite_terrain.keys()):
    if cv_name in resultats:
        resultats_evaluation[cv_name] = evaluer_cv(
            resultats[cv_name], verite_terrain[cv_name]
        )

# Calculer le score global
score_global = calculer_score_global(resultats_evaluation)

print("Evaluation terminee pour", len(resultats_evaluation), "CV")

### Resultats de l'evaluation

In [ ]:
# Afficher les resultats par CV
champs = ["nom", "email", "telephone", "competences",
          "experience_professionnelle", "formation", "langues"]

for cv_name in sorted(resultats_evaluation.keys()):
    scores = resultats_evaluation[cv_name]
    print(f"\n{'='*60}")
    print(f"  {cv_name}")
    print(f"{'='*60}")
    print(f"{'Champ':<30s} {'Precision':>10s} {'Rappel':>10s} {'F1':>10s}")
    print("-" * 62)
    for champ in champs:
        if champ in scores:
            m = scores[champ]
            print(f"{champ:<30s} {m['precision']:>10.2f} {m['rappel']:>10.2f} {m['f1']:>10.2f}")

In [ ]:
# Score global
print("\n" + "=" * 60)
print("SCORE GLOBAL DE QUALITE")
print("=" * 60)
print(f"  Precision moyenne : {score_global['precision_moyenne']:.4f}")
print(f"  Rappel moyen      : {score_global['rappel_moyen']:.4f}")
print(f"  F1-Score moyen    : {score_global['f1_moyen']:.4f}")
print("=" * 60)

# Interpretation
f1 = score_global['f1_moyen']
if f1 >= 0.95:
    qualite = "Excellente"
elif f1 >= 0.85:
    qualite = "Tres bonne"
elif f1 >= 0.70:
    qualite = "Bonne"
elif f1 >= 0.50:
    qualite = "Moyenne"
else:
    qualite = "Insuffisante"

print(f"\nQualite globale de l'extraction : {qualite}")

### Analyse des resultats

L'evaluation montre que notre extracteur base sur les expressions regulieres
obtient de tres bons resultats :

- **Champs simples** (nom, email, telephone) : extraction parfaite (F1 = 1.0)
  grace aux patterns regex bien definis.

- **Competences** : leger deficit de precision du a la segmentation des elements
  contenant des parentheses (ex: "AWS (S3, SageMaker, EC2)" est decoupe en
  plusieurs elements au lieu d'un seul).

- **Experience et formation** : extraction parfaite des elements structurels
  (periode, poste, entreprise, diplome, etablissement).

- **Langues** : extraction parfaite des paires langue/niveau.

Les principales limites identifiees concernent le parsing des competences
lorsque les elements contiennent des virgules dans des sous-groupes entre
parentheses. Une amelioration possible serait d'utiliser un parsing plus
intelligent qui respecte les groupes entre parentheses.

## Conclusion

Ce notebook a demontre comment extraire des informations structurees depuis
un CV au format texte brut en utilisant uniquement des expressions regulieres.

**Points cles :**
- Les regex sont efficaces pour des formats semi-structures
- La detection de sections permet de delimiter les zones d'extraction
- L'approche fonctionne pour le francais et l'anglais
- L'evaluation quantitative confirme une qualite globale elevee (F1 > 0.98)
- Les erreurs residuelles concernent la segmentation des competences avec parentheses
- Pour plus de robustesse, spaCy ou des transformers seraient recommandes